In [1]:
from pathlib import Path
import pandas as pd

# Works whether the notebook runs from the project root or from notebooks/
DATA_DIR = Path("data")
if not DATA_DIR.exists():
    DATA_DIR = Path("../data")

train_path = DATA_DIR / "DeepX_train.csv"
validation_path = DATA_DIR / "DeepX_validation.csv"
unlabeled_path = DATA_DIR / "DeepX_unlabeled.csv"

for file_path in [train_path, validation_path, unlabeled_path]:
    if not file_path.exists():
        raise FileNotFoundError(f"Missing data file: {file_path.resolve()}")

train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)
unlabeled_df = pd.read_csv(unlabeled_path)

print(f"Train shape: {train_df.shape}")
print(f"Validation shape: {validation_df.shape}")
print(f"Unlabeled shape: {unlabeled_df.shape}")

train_df.head()

Train shape: (1971, 9)
Validation shape: (500, 9)
Unlabeled shape: (7047, 7)


,review_id,review_text,star_rating,date,business_name,business_category,platform,aspects,aspect_sentiments
0,7238,لا يوجد الدفع بالبطاقه عند الاستلام,3,2026-03-08 00:00:00,Noon,ecommerce,play_store,"[""app_experience"", ""delivery""]","{""app_experience"": ""negative"", ""delivery"": ""ne..."
1,1036,المكان نضيف وجميل وقعدته تحفه والخدمة فوق المم...,5,قبل يومين (2),ممشي مصر Mawlana Cafe,كافيه,google_maps,"[""cleanliness"", ""ambiance"", ""service""]","{""cleanliness"": ""positive"", ""ambiance"": ""posit..."
2,1975,تجربة سيئة سألتهم الاكل هياخد وقت قد ايه قالول...,1,قبل شهر,بيت لحم Beet Lahm,مطعم,google_maps,"[""service"", ""delivery"", ""food""]","{""service"": ""negative"", ""delivery"": ""negative""..."
3,3024,احلي مكان فزايد,5,قبل شهر,ذا بلكون كافيه الشيخ زايد,مطعم مأكولات ومشروبات,google_maps,"[""general""]","{""general"": ""positive""}"
4,5483,الفطير حلو جدا\nالاحجام تحفة\nبالنسبه للسعر فا...,4,قبل سنة,The Best Restaurant,مطعم,google_maps,"[""food"", ""price""]","{""food"": ""positive"", ""price"": ""positive""}"


In [2]:
train_df.drop(columns=['platform'])
validation_df.drop(columns=['platform'])
unlabeled_df.drop(columns=['platform'])

,review_id,review_text,star_rating,date,business_name,business_category
0,1,Incroyablement grand avec des belles boutiques...,5,قبل 7 ساعات,مول سيتي ستارز.,مركز تسوق
1,2,زحمه جدا,5,قبل 12 ساعة,مول سيتي ستارز.,مركز تسوق
2,3,حلو فخم كشخة محترم ورايق ينفع للعوائل الخليجي...,5,قبل يوم واحد,مول سيتي ستارز.,مركز تسوق
3,4,طبعا غني عن التعريف بتاع البشوات,5,قبل يوم واحد,مول سيتي ستارز.,مركز تسوق
4,5,Centro commerciale al Cairo... Molto grande e ...,5,قبل يومين (2),مول سيتي ستارز.,مركز تسوق
...,...,...,...,...,...,...
7042,10009,تطبيق ممتاز وسهل ويوفر حجوزات للفنادق والطيران...,5,2025-11-22,Booking,travel
7043,10011,غرف المستقبل غير نضاميه,1,2025-11-21,Booking,travel
7044,10012,جدا رائع,5,2025-11-21,Booking,travel
7045,10014,شكرا 🌹🌹🌹🌹,5,2025-11-20,Booking,travel


In [3]:
train_df['business_name'].value_counts()

business_name
Aqarmap                     100
Careem                       87
Elmenus                      82
Spotify                      64
Booking                      62
                           ... 
Brunch & Cake Lake View       1
سوبر ماركت القدس              1
فيرست مول                     1
سعودى سوبر ماركت الزمالك      1
Favilla Lounge                1
Name: count, Length: 388, dtype: int64

In [4]:
aspects_taxanomy = ["food", "service", "price", "cleanliness", "delivery", "ambiance", "app_experience", "general", "none"]

In [5]:
import ast
import json

def parse_aspects(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return [str(v).strip() for v in value if str(v).strip()]
    if not isinstance(value, str):
        text = str(value).strip()
        return [text] if text else []

    value = value.strip()
    if not value or value == "[]":
        return []

    for parser in (json.loads, ast.literal_eval):
        try:
            parsed = parser(value)
            if isinstance(parsed, list):
                return [str(v).strip() for v in parsed if str(v).strip()]
        except Exception:
            pass

    value = value.strip("[]")
    return [item.strip().strip("\"'") for item in value.split(",") if item.strip().strip("\"'")]

def one_hot_encode_aspects(df, taxonomy):
    if "aspects" not in df.columns:
        raise KeyError(f"'aspects' column not found. Available columns: {df.columns.tolist()}")

    parsed = df["aspects"].apply(parse_aspects)
    exploded = parsed.explode()

    if exploded.dropna().empty:
        encoded = pd.DataFrame(0, index=df.index, columns=taxonomy)
    else:
        encoded = exploded.str.get_dummies().groupby(level=0).max()
        encoded = encoded.reindex(columns=taxonomy, fill_value=0)

    encoded = encoded.add_prefix("aspect__")
    return pd.concat([df, encoded], axis=1)

train_df_ohe = one_hot_encode_aspects(train_df, aspects_taxanomy)
validation_df_ohe = one_hot_encode_aspects(validation_df, aspects_taxanomy)

train_df_ohe.filter(like="aspect__").head()

,aspect__food,aspect__service,aspect__price,aspect__cleanliness,aspect__delivery,aspect__ambiance,aspect__app_experience,aspect__general,aspect__none
0,0,0,0,0,1,0,1,0,0
1,0,1,0,1,0,1,0,0,0
2,1,1,0,0,1,0,0,0,0
3,0,0,0,0,0,0,0,1,0
4,1,0,1,0,0,0,0,0,0


In [6]:
SENTIMENT_MAP = {
    "negative": -1,
    "neg": -1,
    "neutral": 0,
    "neu": 0,
    "positive": 1,
    "pos": 1,
}

def parse_aspect_sentiments(value):
    if pd.isna(value):
        return {}
    if isinstance(value, dict):
        return {str(k).strip().lower(): str(v).strip().lower() for k, v in value.items()}
    if not isinstance(value, str):
        return {}

    text = value.strip()
    if not text:
        return {}

    # Handles CSV-escaped payloads like "{""cleanliness"": ""positive""}"
    text = text.replace('""', '"')

    def _try_parse(payload):
        for parser in (json.loads, ast.literal_eval):
            try:
                return parser(payload)
            except Exception:
                pass
        return None

    parsed = _try_parse(text)
    if isinstance(parsed, str):
        parsed = _try_parse(parsed.strip())

    if not isinstance(parsed, dict):
        return {}

    return {str(k).strip().lower(): str(v).strip().lower() for k, v in parsed.items()}

def add_aspect_sentiment_labels(df, taxonomy, source_col="aspect_sentiments", missing_value=0):
    if source_col not in df.columns:
        raise KeyError(f"'{source_col}' column not found. Available columns: {df.columns.tolist()}")

    taxonomy_normalized = [str(aspect).strip().lower() for aspect in taxonomy]
    label_cols = [f"sentiment__{aspect}" for aspect in taxonomy_normalized]

    def encode_row(value):
        sentiments = parse_aspect_sentiments(value)
        row = {col: missing_value for col in label_cols}
        for aspect, sentiment in sentiments.items():
            col = f"sentiment__{aspect}"
            if col in row:
                row[col] = SENTIMENT_MAP.get(str(sentiment).strip().lower(), missing_value)
        return row

    labels_df = df[source_col].apply(encode_row).apply(pd.Series).astype(int)
    return pd.concat([df, labels_df], axis=1)

train_df_ohe = add_aspect_sentiment_labels(train_df_ohe, aspects_taxanomy)
validation_df_ohe = add_aspect_sentiment_labels(validation_df_ohe, aspects_taxanomy)

sentiment_cols = [f"sentiment__{aspect}" for aspect in aspects_taxanomy]
train_df_ohe[sentiment_cols].head()

,sentiment__food,sentiment__service,sentiment__price,sentiment__cleanliness,sentiment__delivery,sentiment__ambiance,sentiment__app_experience,sentiment__general,sentiment__none
0,0,0,0,0,-1,0,-1,0,0
1,0,1,0,1,0,1,0,0,0
2,0,-1,0,0,-1,0,0,0,0
3,0,0,0,0,0,0,0,1,0
4,1,0,1,0,0,0,0,0,0


In [7]:
def aspect_sentiment_summary(df, taxonomy, aspect_prefix="aspect__", sentiment_prefix="sentiment__"):
    rows = []
    for aspect in taxonomy:
        aspect_key = str(aspect).strip().lower()
        aspect_col = f"{aspect_prefix}{aspect_key}"
        sentiment_col = f"{sentiment_prefix}{aspect_key}"

        if aspect_col not in df.columns or sentiment_col not in df.columns:
            rows.append({
                "aspect": aspect_key,
                "aspect_count": 0,
                "neg_count": 0,
                "neu_count": 0,
                "pos_count": 0,
            })
            continue

        has_aspect = df[aspect_col] == 1
        sentiments = df.loc[has_aspect, sentiment_col]

        rows.append({
            "aspect": aspect_key,
            "aspect_count": int(has_aspect.sum()),
            "neg_count": int((sentiments == -1).sum()),
            "neu_count": int((sentiments == 0).sum()),
            "pos_count": int((sentiments == 1).sum()),
        })

    summary = pd.DataFrame(rows).sort_values("aspect_count", ascending=False).reset_index(drop=True)
    total_aspect_mentions = int(summary["aspect_count"].sum())
    aspects_present = int((summary["aspect_count"] > 0).sum())
    return summary, total_aspect_mentions, aspects_present

train_summary, train_total_mentions, train_aspects_present = aspect_sentiment_summary(train_df_ohe, aspects_taxanomy)
validation_summary, validation_total_mentions, validation_aspects_present = aspect_sentiment_summary(validation_df_ohe, aspects_taxanomy)

labeled_df_ohe = pd.concat([train_df_ohe, validation_df_ohe], ignore_index=True)
overall_summary, overall_total_mentions, overall_aspects_present = aspect_sentiment_summary(labeled_df_ohe, aspects_taxanomy)

print(f"Train - total aspect mentions: {train_total_mentions}, aspects present: {train_aspects_present}/{len(aspects_taxanomy)}")
print(f"Validation - total aspect mentions: {validation_total_mentions}, aspects present: {validation_aspects_present}/{len(aspects_taxanomy)}")
print(f"Overall - total aspect mentions: {overall_total_mentions}, aspects present: {overall_aspects_present}/{len(aspects_taxanomy)}")

overall_summary

Train - total aspect mentions: 3333, aspects present: 9/9
Validation - total aspect mentions: 840, aspects present: 9/9
Overall - total aspect mentions: 4173, aspects present: 9/9


,aspect,aspect_count,neg_count,neu_count,pos_count
0,service,1241,551,10,680
1,app_experience,573,394,26,153
2,food,556,217,40,299
3,ambiance,478,124,11,343
4,price,434,293,13,128
5,general,377,42,17,318
6,cleanliness,236,91,1,144
7,delivery,209,183,2,24
8,none,69,0,69,0


In [8]:
import difflib
import html
from functools import lru_cache

import requests

MANUAL_CATEGORY_FIXES = {
    "resturant": "restaurant",
    "restuarant": "restaurant",
    "restraunt": "restaurant",
    "caffee": "cafe",
    "cofee": "coffee",
    "bevarages": "beverages",
    "dessertes": "desserts",
}

def normalize_category(value):
    if pd.isna(value):
        return ""
    return " ".join(str(value).strip().lower().split())

def build_spelling_map(series_list, min_canonical_freq=5, similarity_cutoff=0.86):
    all_values = pd.concat([series.map(normalize_category) for series in series_list], ignore_index=True)
    counts = all_values[all_values != ""].value_counts()

    canonical_pool = set(counts[counts >= min_canonical_freq].index.tolist())
    canonical_pool.update(MANUAL_CATEGORY_FIXES.values())
    canonical_pool = sorted(canonical_pool)

    spelling_map = {}
    issue_rows = []

    for original, count in counts.items():
        corrected = MANUAL_CATEGORY_FIXES.get(original, original)

        if corrected == original and original not in canonical_pool and canonical_pool:
            candidate = difflib.get_close_matches(original, canonical_pool, n=1, cutoff=similarity_cutoff)
            if candidate:
                corrected = candidate[0]

        spelling_map[original] = corrected
        if corrected != original:
            issue_rows.append({
                "original": original,
                "suggested": corrected,
                "count": int(count),
            })

    if issue_rows:
        issues_df = pd.DataFrame(issue_rows).sort_values("count", ascending=False).reset_index(drop=True)
    else:
        issues_df = pd.DataFrame(columns=["original", "suggested", "count"])

    return spelling_map, counts, issues_df

@lru_cache(maxsize=2048)
def translate_to_arabic_api(text):
    if not text:
        return ""

    if any("\u0600" <= ch <= "\u06FF" for ch in text):
        return text

    url = "https://api.mymemory.translated.net/get"
    params = {"q": text, "langpair": "en|ar"}

    try:
        response = requests.get(url, params=params, timeout=15)
        response.raise_for_status()
        payload = response.json()
        translated = payload.get("responseData", {}).get("translatedText", "").strip()
        return html.unescape(translated) if translated else text
    except Exception:
        return text

def apply_business_category_preprocessing(df, spelling_map, translation_map):
    processed = df.copy()
    processed["business_category_normalized"] = processed["business_category"].map(normalize_category)
    processed["business_category_clean"] = processed["business_category_normalized"].map(
        lambda x: spelling_map.get(x, x)
    )
    processed["business_category_spelling_issue"] = (
        processed["business_category_clean"] != processed["business_category_normalized"]
    )
    processed["business_category_ar"] = processed["business_category_clean"].map(
        lambda x: translation_map.get(x, "")
    )
    return processed

series_for_spelling = [train_df_ohe["business_category"], validation_df_ohe["business_category"]]
if "business_category" in unlabeled_df.columns:
    series_for_spelling.append(unlabeled_df["business_category"])

spelling_map, train_category_counts, train_category_issues = build_spelling_map(series_for_spelling)

all_clean_categories = sorted(set(spelling_map.values()))
category_translation_map = {
    category: translate_to_arabic_api(category)
    for category in all_clean_categories
}

train_df_ohe = apply_business_category_preprocessing(train_df_ohe, spelling_map, category_translation_map)
validation_df_ohe = apply_business_category_preprocessing(validation_df_ohe, spelling_map, category_translation_map)

if "business_category" in unlabeled_df.columns:
    unlabeled_df = apply_business_category_preprocessing(unlabeled_df, spelling_map, category_translation_map)

print(f"Detected possible spelling issues in unique categories: {len(train_category_issues)}")
print(f"Unique cleaned categories translated via API: {len(all_clean_categories)}")

train_translation_preview = (
    train_df_ohe[[
        "business_category",
        "business_category_clean",
        "business_category_ar",
        "business_category_spelling_issue",
    ]]
    .drop_duplicates()
    .sort_values(["business_category_spelling_issue", "business_category"]).reset_index(drop=True)
 )

display(train_category_issues.head(30))
display(train_translation_preview.head(30))

Detected possible spelling issues in unique categories: 0
Unique cleaned categories translated via API: 71


,original,suggested,count


,business_category,business_category_clean,business_category_ar,business_category_spelling_issue
0,ecommerce,ecommerce,التجارة الإلكترونية,False
1,entertainment,entertainment,الترفيه,False
2,food_delivery,food_delivery,تطبيقات توصيل الطلبات,False
3,real_estate,real_estate,الأرض وما عليها من إنشاءات.,False
4,transport,transport,الموصلات والنقل,False
5,travel,travel,السفر,False
6,المستشفى الحكومي,المستشفى الحكومي,المستشفى الحكومي,False
7,المستشفى العسكري,المستشفى العسكري,المستشفى العسكري,False
8,برنامج اللياقة البدنية,برنامج اللياقة البدنية,برنامج اللياقة البدنية,False
9,سوبرماركت,سوبرماركت,سوبرماركت,False


In [11]:
train_df_ohe.head()
columns = ['business_category','aspects','aspect_sentiments','business_category_normalized','business_category_clean','business_category_spelling_issue','platform']
train_df_ohe.drop(columns=columns)
validation_df_ohe.drop(columns=columns)

,review_id,review_text,star_rating,date,business_name,aspect__food,aspect__service,aspect__price,aspect__cleanliness,aspect__delivery,...,sentiment__food,sentiment__service,sentiment__price,sentiment__cleanliness,sentiment__delivery,sentiment__ambiance,sentiment__app_experience,sentiment__general,sentiment__none,business_category_ar
0,4446,مريم سوتلي الاظافررر تحفههه اوييي ❤️❤️❤️❤️❤️,5,قبل شهرين,Sand salon,0,1,0,0,0,...,0,1,0,0,0,0,0,0,0,صالون تجميل
1,8612,التطبيق جميل .. أتمنى إضافة البحث عن طريق الخر...,4,2020-10-28 00:00:00,Aqarmap,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,الأرض وما عليها من إنشاءات.
2,6729,سراقين مكتوب وصلت السياره والسواق مارضى يقول و...,1,2026-02-04 00:00:00,Careem,0,1,1,0,1,...,0,-1,-1,0,-1,0,0,0,0,الموصلات والنقل
3,6292,سي جيدا,1,2025-08-07 00:00:00,Elmenus,0,0,0,0,0,...,0,0,0,0,0,0,0,-1,0,تطبيقات توصيل الطلبات
4,1639,مكان ممتاز جدا و الخدمة جيده جدا,4,قبل أسبوع,Holm Cafe,0,1,0,0,0,...,0,1,0,0,0,1,0,0,0,مقهى
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,6460,ولا في قسم مساعده واوردر بيتأخى دايما عن ميعاد...,1,2023-12-09 00:00:00,Elmenus,0,1,0,0,1,...,0,-1,0,0,-1,0,0,0,0,تطبيقات توصيل الطلبات
496,6234,تطبيق معفن الاعلانات كلها غلط و نصب و يقعدو يع...,1,2026-02-04 00:00:00,Elmenus,0,1,1,0,0,...,0,-1,-1,0,0,0,-1,0,0,تطبيقات توصيل الطلبات
497,9593,جيد جدا سهوله فى الحجز موفر للوقت,5,2026-01-16 00:00:00,Wego,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,السفر
498,3686,ماشالله تسلم ايديها سريعه وعملتلي شعري حلو اوي,5,قبل أسبوع,كوافير ندى,0,1,0,0,0,...,0,1,0,0,0,0,0,0,0,صالون تجميل


In [16]:
import os
import re
from functools import lru_cache

import requests

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

ARABIC_DIGIT_MAP = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
UNIT_TO_DAYS = {
    "minute": 1 / 1440,
    "hour": 1 / 24,
    "day": 1,
    "week": 7,
    "month": 30,
    "year": 365,
}

AR_NUMBER_WORDS = {
    "واحد": 1,
    "واحدة": 1,
    "اثنين": 2,
    "اثنان": 2,
    "اثنتين": 2,
    "اثنتان": 2,
    "ثلاثة": 3,
    "ثلاث": 3,
    "اربعة": 4,
    "أربعة": 4,
    "خمسة": 5,
    "ستة": 6,
    "سبعة": 7,
    "ثمانية": 8,
    "تسعة": 9,
    "عشرة": 10,
    "احد عشر": 11,
    "إحدى عشرة": 11,
    "اثنا عشر": 12,
}

def normalize_arabic_date_text(text):
    normalized = str(text).strip().lower().translate(ARABIC_DIGIT_MAP)
    normalized = re.sub(r"[\u064b-\u0652]", "", normalized)
    normalized = normalized.replace("،", " ")
    normalized = re.sub(r"^تاريخ التعديل\s*[:：]?\s*", "", normalized)
    normalized = re.sub(r"\(\s*\d+\s*\)", "", normalized)
    normalized = re.sub(r"\s+", " ", normalized).strip()
    return normalized

def _to_int_number(raw):
    raw = raw.strip()
    if raw.isdigit():
        return int(raw)
    return AR_NUMBER_WORDS.get(raw)

def local_relative_days_parser(text):
    normalized = normalize_arabic_date_text(text)

    direct_map = {
        "الان": 0,
        "الآن": 0,
        "اليوم": 0,
        "امس": 1,
        "أمس": 1,
        "اول امس": 2,
        "أول أمس": 2,
        "قبل ساعة": 0,
        "قبل ساعتين": 0,
        "قبل يوم": 1,
        "قبل يوم واحد": 1,
        "قبل يومين": 2,
        "قبل اسبوع": 7,
        "قبل أسبوع": 7,
        "قبل اسبوعين": 14,
        "قبل أسبوعين": 14,
        "قبل شهر": 30,
        "قبل شهرين": 60,
        "قبل سنة": 365,
        "قبل سنتين": 730,
        "منذ ساعة": 0,
        "منذ ساعتين": 0,
        "منذ يوم": 1,
        "منذ يوم واحد": 1,
        "منذ يومين": 2,
        "منذ اسبوع": 7,
        "منذ أسبوع": 7,
        "منذ اسبوعين": 14,
        "منذ أسبوعين": 14,
        "منذ شهر": 30,
        "منذ شهرين": 60,
        "منذ سنة": 365,
        "منذ سنتين": 730,
    }

    if normalized in direct_map:
        return direct_map[normalized]

    numeric_or_word_pattern = re.compile(
        r"(?:قبل|منذ)\s+(\d+|واحد|واحدة|اثنين|اثنان|اثنتين|اثنتان|ثلاثة|ثلاث|اربعة|أربعة|خمسة|ستة|سبعة|ثمانية|تسعة|عشرة|احد عشر|إحدى عشرة|اثنا عشر)\s*(دقيقة|دقائق|ساعة|ساعات|يوم|ايام|أيام|اسبوع|أسبوع|اسابيع|أسابيع|شهر|شهور|أشهر|سنة|سنه|سنوات)",
        flags=re.IGNORECASE,
    )
    match = numeric_or_word_pattern.search(normalized)
    if match:
        count = _to_int_number(match.group(1))
        unit_raw = match.group(2)
        if count is None:
            return None

        if unit_raw in {"دقيقة", "دقائق"}:
            return int(count * UNIT_TO_DAYS["minute"])
        if unit_raw in {"ساعة", "ساعات"}:
            return int(count * UNIT_TO_DAYS["hour"])
        if unit_raw in {"يوم", "ايام", "أيام"}:
            return int(count * UNIT_TO_DAYS["day"])
        if unit_raw in {"اسبوع", "أسبوع", "اسابيع", "أسابيع"}:
            return int(count * UNIT_TO_DAYS["week"])
        if unit_raw in {"شهر", "شهور", "أشهر"}:
            return int(count * UNIT_TO_DAYS["month"])
        if unit_raw in {"سنة", "سنه", "سنوات"}:
            return int(count * UNIT_TO_DAYS["year"])

    return None

@lru_cache(maxsize=4096)
def relative_days_with_chatgpt(text, today_iso):
    if not OPENAI_API_KEY:
        return None

    system_prompt = (
        "You convert Arabic relative date phrases to integer days ago. "
        "Reference date is provided as today's date. "
        "Return only one integer with no explanation. "
        "If the phrase cannot be interpreted, return -999999."
    )
    user_prompt = (
        f"Reference date (today): {today_iso}. "
        f"Phrase: {text}. "
        "How many days ago is this phrase? Return only an integer."
    )

    try:
        response = requests.post(
            "https://api.openai.com/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {OPENAI_API_KEY}",
                "Content-Type": "application/json",
            },
            json={
                "model": OPENAI_MODEL,
                "temperature": 0,
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
            },
            timeout=30,
        )
        response.raise_for_status()
        content = response.json()["choices"][0]["message"]["content"].strip()
        match = re.search(r"-?\d+", content)
        if not match:
            return None
        parsed = int(match.group())
        return None if parsed == -999999 else parsed
    except Exception:
        return None

def parse_mixed_date_to_days_since_today(value, today_date):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()
    if not text:
        return pd.NA

    absolute_dt = pd.to_datetime(text, errors="coerce")
    if pd.notna(absolute_dt):
        return int((today_date - absolute_dt.normalize()).days)

    normalized_text = normalize_arabic_date_text(text)

    # Prefer ChatGPT parsing when API key is available.
    gpt_days = relative_days_with_chatgpt(normalized_text, str(today_date.date()))
    if gpt_days is not None:
        return int(gpt_days)

    local_days = local_relative_days_parser(normalized_text)
    if local_days is not None:
        return int(local_days)

    return pd.NA

def add_days_since_today_column(df, date_col="date", output_col="date_days_since_today", today_date=None):
    if date_col not in df.columns:
        raise KeyError(f"'{date_col}' not found. Available columns: {df.columns.tolist()}")

    if today_date is None:
        today_date = pd.Timestamp.today().normalize()
    else:
        today_date = pd.to_datetime(today_date).normalize()

    processed = df.copy()
    processed[output_col] = processed[date_col].apply(
        lambda x: parse_mixed_date_to_days_since_today(x, today_date)
    )
    return processed

reference_today = pd.Timestamp.today().normalize()
train_df_ohe = add_days_since_today_column(train_df_ohe, today_date=reference_today)
validation_df_ohe = add_days_since_today_column(validation_df_ohe, today_date=reference_today)

if "date" in unlabeled_df.columns:
    unlabeled_df = add_days_since_today_column(unlabeled_df, today_date=reference_today)

reports = {}
for name, frame in {"train": train_df_ohe, "validation": validation_df_ohe}.items():
    parsed_count = int(frame["date_days_since_today"].notna().sum())
    failed_count = int(frame["date_days_since_today"].isna().sum())
    reports[name] = {
        "rows": int(len(frame)),
        "parsed": parsed_count,
        "failed": failed_count,
    }

if not OPENAI_API_KEY:
    print("OPENAI_API_KEY is not set. Local parser was used; ChatGPT API fallback is disabled.")
else:
    print(f"ChatGPT API model used for parsing: {OPENAI_MODEL}")

print(f"Reference today date: {reference_today.date()}")
display(pd.DataFrame(reports).T)

failed_preview = train_df_ohe.loc[
    train_df_ohe["date_days_since_today"].isna(),
    ["date"],
][:20]

display(train_df_ohe[["date", "date_days_since_today"]].head(20))
display(failed_preview)

OPENAI_API_KEY is not set. Local parser was used; ChatGPT API fallback is disabled.
Reference today date: 2026-04-24


,rows,parsed,failed
train,1971,1971,0
validation,500,500,0


,date,date_days_since_today
0,2026-03-08 00:00:00,47
1,قبل يومين (2),2
2,قبل شهر,30
3,قبل شهر,30
4,قبل سنة,365
5,2024-08-17 00:00:00,615
6,قبل 3 أسابيع,21
7,2026-03-02 00:00:00,53
8,قبل أسبوعين,14
9,قبل 11 شهرًا,330


,date
